<a href="https://colab.research.google.com/github/giovanep4mg/-7DaysOfCode_Primeiro_Dia/blob/main/caixa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Instala as dependências necessárias no Colab
!pip install pandas xlsxwriter > /dev/null 2>&1

import pandas as pd
import os

# Nome padrão do arquivo de testes no Colab
NOME_ARQUIVO = "caixa_teste.xlsx"

def criar_ou_carregar_excel():
    if os.path.exists(NOME_ARQUIVO):
        df = pd.read_excel(NOME_ARQUIVO)
        print(f"\n📁 Arquivo '{NOME_ARQUIVO}' carregado com sucesso! ({len(df)} registros encontrados).")
        return df
    else:
        dados = {
            'Dia': [], 'Dinh/Salao': [], 'Notas2': [], 'Moeda/Salao': [], 'Dinh/Casa': [],
            'Moeda/Casa': [], 'Gasto/Dia': [], 'TotalDia': [], 'Sicoob': [], 'Sumup': [],
            'Nullbank': [], 'MercPago': [], 'TotalBancos': [], 'Caixa': [], 'Casa': [],
            'TotalSoma': [], 'Lucro': [], 'TotalAnterior': []
        }
        df = pd.DataFrame(dados)
        print(f"\n✨ Novo arquivo '{NOME_ARQUIVO}' criado com a estrutura padrão.")
        return df

def obter_valor_numerico(mensagem):
    while True:
        valor = input(f"{mensagem}: ").strip()
        if valor == "":
            return 0.0
        try:
            return float(valor)
        except ValueError:
            print("❌ Por favor, insira um valor numérico válido.")

def simular_cadastro():
    df = criar_ou_carregar_excel()

    print("\n" + "="*30)
    print("   INICIANDO NOVO LANÇAMENTO")
    print("="*30)

    dia_val = input("Digite o dia: ").strip()
    if dia_val == "":
        print("Operação cancelada.")
        return df
    dia = int(float(dia_val))

    dinhSalao = obter_valor_numerico("Dinheiro Salão")
    notas2 = obter_valor_numerico("Notas de 2")
    moedaSalao = obter_valor_numerico("Moeda Salão")
    dinhCasa = obter_valor_numerico("Dinheiro Casa")
    moedaCasa = obter_valor_numerico("Moeda Casa")
    gastoDia = obter_valor_numerico("Gasto Dia")

    # --- AUTOMAÇÃO DOS BANCOS (Mantém o anterior se deixar em branco) ---
    def pegar_valor_banco(nome_coluna, mensagem_prompt):
        valor_digitado = input(f"{mensagem_prompt} (Deixe em branco para manter o anterior): ").strip()

        if valor_digitado == "" and not df.empty and nome_coluna in df.columns:
            valor_anterior = df[nome_coluna].iloc[-1]
            print(100 * " " + f"➡️ Mantendo valor anterior: {valor_anterior}")
            return valor_anterior

        if valor_digitado == "":
            return 0.0

        try:
            return float(valor_digitado)
        except ValueError:
            print("❌ Valor inválido. Usando o valor anterior ou 0.")
            if not df.empty and nome_coluna in df.columns:
                return df[nome_coluna].iloc[-1]
            return 0.0

    print("\n--- ATUALIZAÇÃO DOS BANCOS ---")
    sicoob = pegar_valor_banco('Sicoob', "Sicoob")
    sumup = pegar_valor_banco('Sumup', "Sumup")
    nullbank = pegar_valor_banco('Nullbank', "Nullbank")
    mercPago = pegar_valor_banco('MercPago', "MercPago")

    casa = dinhCasa
    novo_moedaCasa = moedaCasa
    caixa = dinhSalao + notas2
    totalbancos = sicoob + sumup + nullbank + mercPago

    # Lógica para Moeda Casa Anterior
    if not df.empty and 'Moeda/Casa' in df.columns:
        moedaCasa_anterior = df['Moeda/Casa'].iloc[-1]
    else:
        moedaCasa_anterior = obter_valor_numerico("Digite o valor anterior de Moeda Casa")

    moedaCasa_atual = novo_moedaCasa + moedaCasa_anterior
    totalsoma = casa + caixa + totalbancos + moedaSalao + moedaCasa_atual

    # Lógica para Total Anterior
    if not df.empty and 'TotalSoma' in df.columns:
        total_anterior = df['TotalSoma'].iloc[-1]
    else:
        total_anterior = obter_valor_numerico("Digite o valor do total anterior")

    novo_lucro = totalsoma - total_anterior
    totalDia = novo_lucro + gastoDia

    nova_linha = {
        'Dia': dia, 'Dinh/Salao': dinhSalao, 'Notas2': notas2, 'Moeda/Salao': moedaSalao,
        'Dinh/Casa': dinhCasa, 'Moeda/Casa': moedaCasa_atual, 'Gasto/Dia': gastoDia,
        'Sicoob': sicoob, 'Sumup': sumup, 'Nullbank': nullbank, 'MercPago': mercPago,
        'TotalBancos': totalbancos, 'Casa': casa, 'Caixa': caixa, 'TotalSoma': totalsoma,
        'Lucro': novo_lucro, 'TotalDia': totalDia, 'TotalAnterior': total_anterior
    }

    df_novo = pd.DataFrame([nova_linha])
    df = pd.concat([df, df_novo], ignore_index=True)

    # Salva no Excel com formatação
    writer = pd.ExcelWriter(NOME_ARQUIVO, engine='xlsxwriter')
    df.to_excel(writer, index=False)
    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    formato_numero = workbook.add_format({'num_format': '#,##0.00'})
    worksheet.set_column('B:S', 15, formato_numero)
    writer.close()

    print(f"\n✅ Dados do dia {dia} salvos com sucesso!")
    return df

# --- EXECUTANDO O TESTE NO COLAB ---
tabela_atualizada = simular_cadastro()

# Exibe a tabela formatada abaixo da célula
display(tabela_atualizada)


📁 Arquivo 'caixa_teste.xlsx' carregado com sucesso! (1 registros encontrados).

   INICIANDO NOVO LANÇAMENTO
Digite o dia: 2
Dinheiro Salão: 2100
Notas de 2: 70
Moeda Salão: 12.50
Dinheiro Casa: 63
Moeda Casa: 0
Gasto Dia: 94.50

--- ATUALIZAÇÃO DOS BANCOS ---
Sicoob (Deixe em branco para manter o anterior): 
                                                                                                    ➡️ Mantendo valor anterior: 26.23
Sumup (Deixe em branco para manter o anterior): 
                                                                                                    ➡️ Mantendo valor anterior: 34.86
Nullbank (Deixe em branco para manter o anterior): 330.88
MercPago (Deixe em branco para manter o anterior): 
                                                                                                    ➡️ Mantendo valor anterior: 1941.73

✅ Dados do dia 2 salvos com sucesso!


,Dia,Dinh/Salao,Notas2,Moeda/Salao,Dinh/Casa,Moeda/Casa,Gasto/Dia,TotalDia,Sicoob,Sumup,Nullbank,MercPago,TotalBancos,Caixa,Casa,TotalSoma,Lucro,TotalAnterior
0,1,2000.0,70.0,11.0,34.0,8.75,79.52,2143.35,26.23,34.86,305.88,1941.73,2308.7,2070.0,34.0,4432.45,2063.83,2368.62
1,2,2100.0,70.0,12.5,63.0,8.75,94.50,250.00,26.23,34.86,330.88,1941.73,2333.7,2170.0,63.0,4587.95,155.50,4432.45


In [ ]:
# 1. Instala as dependências necessárias no Colab
!pip install pandas xlsxwriter > /dev/null 2>&1

import pandas as pd
import os

# Nome padrão do arquivo de testes no Colab
NOME_ARQUIVO = "caixa_teste.xlsx"

def criar_ou_carregar_excel():
    if os.path.exists(NOME_ARQUIVO):
        df = pd.read_excel(NOME_ARQUIVO)
        print(f"\n📁 Arquivo '{NOME_ARQUIVO}' carregado com sucesso! ({len(df)} registros encontrados).")
        return df
    else:
        dados = {
            'Dia': [], 'Dinh/Salao': [], 'Notas2': [], 'Moeda/Salao': [], 'Dinh/Casa': [],
            'Moeda/Casa': [], 'Gasto/Dia': [], 'TotalDia': [], 'Sicoob': [], 'Sumup': [],
            'Nullbank': [], 'MercPago': [], 'TotalBancos': [], 'Caixa': [], 'Casa': [],
            'TotalSoma': [], 'Lucro': [], 'TotalAnterior': []
        }
        df = pd.DataFrame(dados)
        print(f"\n✨ Novo arquivo '{NOME_ARQUIVO}' criado com a estrutura padrão.")
        return df

def obter_valor_numerico(mensagem):
    while True:
        valor = input(f"{mensagem}: ").strip()

        if valor == "":
            return 0.0

        # 💡 TRATAMENTO: Substitui a vírgula por ponto para aceitar ambos os formatos
        valor_tratado = valor.replace(',', '.')

        try:
            return float(valor_tratado)
        except ValueError:
            print("❌ Valor inválido! Por favor, utilize apenas números (ex: 13.45 ou 13,45). Tente novamente.")

def simular_cadastro():
    df = criar_ou_carregar_excel()

    print("\n" + "="*30)
    print("   INICIANDO NOVO LANÇAMENTO")
    print("="*30)

    dia_val = input("Digite o dia: ").strip()
    if dia_val == "":
        print("Operação cancelada.")
        return df
    dia = int(float(dia_val.replace(',', '.')))

    dinhSalao = obter_valor_numerico("Dinheiro Salão")
    notas2 = obter_valor_numerico("Notas de 2")
    moedaSalao = obter_valor_numerico("Moeda Salão")
    dinhCasa = obter_valor_numerico("Dinheiro Casa")
    moedaCasa = obter_valor_numerico("Moeda Casa")
    gastoDia = obter_valor_numerico("Gasto Dia")

    # --- AUTOMAÇÃO DOS BANCOS (Trata vírgula e mantém o anterior se deixar em branco) ---
    def pegar_valor_banco(nome_coluna, mensagem_prompt):
        while True:
            valor_digitado = input(f"{mensagem_prompt} (Deixe em branco para manter o anterior): ").strip()

            if valor_digitado == "" and not df.empty and nome_coluna in df.columns:
                valor_anterior = df[nome_coluna].iloc[-1]
                print(f"➡️ Mantendo valor anterior: {valor_anterior}")
                return valor_anterior

            if valor_digitado == "":
                return 0.0

            # 💡 TRATAMENTO DA VÍRGULA NOS BANCOS
            valor_tratado = valor_digitado.replace(',', '.')

            try:
                return float(valor_tratado)
            except ValueError:
                print("❌ Valor inválido! Use ponto ou vírgula para decimais. Tente novamente.")

    print("\n--- ATUALIZAÇÃO DOS BANCOS ---")
    sicoob = pegar_valor_banco('Sicoob', "Sicoob")
    sumup = pegar_valor_banco('Sumup', "Sumup")
    nullbank = pegar_valor_banco('Nullbank', "Nullbank")
    mercPago = pegar_valor_banco('MercPago', "MercPago")

    casa = dinhCasa
    novo_moedaCasa = moedaCasa
    caixa = dinhSalao + notas2
    totalbancos = sicoob + sumup + nullbank + mercPago

    # Lógica para Moeda Casa Anterior
    if not df.empty and 'Moeda/Casa' in df.columns:
        moedaCasa_anterior = df['Moeda/Casa'].iloc[-1]
    else:
        moedaCasa_anterior = obter_valor_numerico("Digite o valor anterior de Moeda Casa")

    moedaCasa_atual = novo_moedaCasa + moedaCasa_anterior
    totalsoma = casa + caixa + totalbancos + moedaSalao + moedaCasa_atual

    # Lógica para Total Anterior
    if not df.empty and 'TotalSoma' in df.columns:
        total_anterior = df['TotalSoma'].iloc[-1]
    else:
        total_anterior = obter_valor_numerico("Digite o valor do total anterior")

    novo_lucro = totalsoma - total_anterior
    totalDia = novo_lucro + gastoDia

    nova_linha = {
        'Dia': dia, 'Dinh/Salao': dinhSalao, 'Notas2': notas2, 'Moeda/Salao': moedaSalao,
        'Dinh/Casa': dinhCasa, 'Moeda/Casa': moedaCasa_atual, 'Gasto/Dia': gastoDia,
        'Sicoob': sicoob, 'Sumup': sumup, 'Nullbank': nullbank, 'MercPago': mercPago,
        'TotalBancos': totalbancos, 'Casa': casa, 'Caixa': caixa, 'TotalSoma': totalsoma,
        'Lucro': novo_lucro, 'TotalDia': totalDia, 'TotalAnterior': total_anterior
    }

    df_novo = pd.DataFrame([nova_linha])
    df = pd.concat([df, df_novo], ignore_index=True)

    # Salva no Excel com formatação
    writer = pd.ExcelWriter(NOME_ARQUIVO, engine='xlsxwriter')
    df.to_excel(writer, index=False)
    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    formato_numero = workbook.add_format({'num_format': '#,##0.00'})
    worksheet.set_column('B:S', 15, formato_numero)
    writer.close()

    print(f"\n✅ Dados do dia {dia} salvos com sucesso!")
    return df

# --- EXECUTANDO O TESTE NO COLAB ---
tabela_atualizada = simular_cadastro()

# Exibe a tabela formatada abaixo da célula
display(tabela_atualizada)